# Two-Tower Recommendation Engine with FAISS & LightGBM Re-Ranking

This notebook demonstrates the end-to-end industrial recommendation pipeline:
1. **Stage 1 (Retrieval):** PyTorch Two-Tower Model + FAISS HNSW Vector Index (< 0.1ms latency)
2. **Stage 2 (Re-Ranking):** LightGBM Gradient Boosted Trees with Seen-Item Deduplication
3. **Cold-Start Handling:** 2-Tier Fallback (Demographic Cluster Vectors & IMDb Bayesian Weighted Ratings)

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('../src'))

import numpy as np
import pandas as pd
import torch
import faiss
import lightgbm as lgb

from data_preprocessing import MovieLens1MPreprocessor
from two_tower_model import TwoTowerModel
from vector_search import VectorSearchEngine
from reranker import LightGBMReranker
from cold_start import BayesianPopularityRanker, DemographicColdStartHandler

print("Environment initialized successfully!")

## 1. Load Preprocessed Data

In [ ]:
prep = MovieLens1MPreprocessor("../data/ml-1m").process()
train_df, test_df = prep.temporal_train_test_split(test_ratio=0.2)
print(f"Users: {prep.num_users:,} | Movies: {prep.num_movies:,}")
print(f"Train Ratings: {len(train_df):,} | Test Ratings: {len(test_df):,}")

## 2. Load Trained Two-Tower Model & FAISS Index

In [ ]:
# Load Two-Tower Neural Network
model = TwoTowerModel(prep.num_users, prep.num_movies, num_genres=18, embedding_dim=64)
model.load_state_dict(torch.load("../models/two_tower.pt", map_location="cpu"))
model.eval()

# Load FAISS HNSW Index
index = faiss.read_index("../models/faiss_hnsw.index")
print(f"Loaded FAISS HNSW index containing {index.ntotal} item vectors (dim: {index.d}).")

## 3. Benchmark Retrieval Latency (HNSW vs Brute-force)

In [ ]:
with torch.no_grad():
    sample_user_features = prep.user_features[:500]
    sample_user_embs = model.user_tower(sample_user_features).cpu().numpy().astype(np.float32)

import time
start = time.perf_counter()
scores, indices = index.search(sample_user_embs, 100)
total_time_ms = (time.perf_counter() - start) * 1000.0

print(f"Total time for 500 queries: {total_time_ms:.2f} ms")
print(f"Average Latency per Query: {total_time_ms / 500:.3f} ms (sub-millisecond!)")

## 4. Interactive Recommendation Demo (with Seen-Item Deduplication)

In [ ]:
USER_ID = 42  # Try changing this to any number between 1 and 6040

user_idx = prep.user2idx[USER_ID]
u_feat = prep.user_features[user_idx]
user_seen = set(train_df[train_df["user_idx"] == user_idx]["movie_idx"].values)

with torch.no_grad():
    u_emb = model.user_tower(u_feat.unsqueeze(0)).cpu().numpy().astype(np.float32)

# Stage 1: FAISS Retrieval (Fetch extra candidates to guarantee 100 novel items after seen-filter)
sim_scores, cand_indices = index.search(u_emb, min(100 + len(user_seen), prep.num_movies))
cand_indices = cand_indices[0]
sim_scores = sim_scores[0]

# Deduplication: Filter out already watched movies
unseen_mask = [m not in user_seen for m in cand_indices]
cand_indices = [m for m, keep in zip(cand_indices, unseen_mask) if keep]
sim_scores = [s for s, keep in zip(sim_scores, unseen_mask) if keep]

# Stage 2: LightGBM Re-Ranker
reranker = LightGBMReranker()
reranker.extract_historical_stats(train_df, prep.movies_meta)
reranker.load("../models/lightgbm_reranker.txt")

final_indices, final_probs = reranker.rerank(user_idx, cand_indices, sim_scores)
final_indices = final_indices[:10]
final_probs = final_probs[:10]

# Display Results
results = []
for rank, (m_idx, prob) in enumerate(zip(final_indices, final_probs), 1):
    m_row = prep.movies_meta[prep.movies_meta["movie_idx"] == m_idx].iloc[0]
    results.append({
        "Rank": rank,
        "Movie Title": m_row["title"],
        "Genres": m_row["genres"],
        "Re-Rank Score": f"{prob:.4f}"
    })

pd.DataFrame(results)

## 5. Cold-Start Fallback Demonstration
Handles new users with zero history:

In [ ]:
bayesian = BayesianPopularityRanker(train_df, prep.movies_meta)
handler = DemographicColdStartHandler(
    user_tower=model.user_tower,
    faiss_index=index,
    movies_meta=prep.movies_meta,
    bayesian_ranker=bayesian
)

# Tier 1: Demographic Warm Fallback (e.g. Female, 18-24)
tier1_recs = handler.recommend(gender_str="F", age_str="18-24", top_k=5)
print("=== TIER 1: Demographic Cluster Recommendations (Female, 18-24) ===")
display(pd.DataFrame(tier1_recs)[["rank", "title", "genres", "score"]])

# Tier 2: Pure Anonymous Bayesian Fallback
tier2_recs = bayesian.get_top_k(top_k=5)
print("\n=== TIER 2: Pure Anonymous Fallback (IMDb Bayesian Weighted Classics) ===")
display(pd.DataFrame(tier2_recs)[["rank", "title", "genres", "score"]])